In [1]:
import json
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
with open("dataset.json", "r") as f:
    data = json.load(f)

In [5]:
contexts = []
questions = []
answers = []

for article in data['data']:
    for para in article['paragraphs']:
        context = para['context']
        for qa in para['qas']:
            question = qa['question']
            answer = qa['answers'][0]['text']

            contexts.append(context)
            questions.append(question)      
            answers.append(answer)

df = pd.DataFrame({
    'context': contexts,
    'question': questions,
    'answer': answers
})

df.head()

,context,question,answer
0,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,Saint Bernadette Soubirous
1,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,a copper statue of Christ
2,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,the Main Building
3,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,a Marian place of prayer and reflection
4,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,a golden statue of the Virgin Mary


In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[a-zA-Z0-9]', '', text)
    return text

df['clean_context'] = df['context'].apply(clean_text)
df['clean_question'] = df['question'].apply(clean_text)

In [7]:
def split_sentences(text):
    sentences = re.split(r'[.!?]+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 0]

In [8]:
def answer_question(context, question):
    sentences = split_sentences(context)
    all_sentences = sentences + [question]
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(all_sentences)
    question_vector = tfidf_matrix[-1]
    similarities = cosine_similarity(question_vector, tfidf_matrix[:-1])
    best_idx = np.argmax(similarities)
    return sentences[best_idx]

In [9]:
for i in range(3):
    context = df['context'][i]
    question = df['question'][i]
    actual = df['answer'][i]
    predicted = answer_question(context, question)

    print("\nQUESTION:", question)
    print("ACTUAL ANSWER:", actual)
    print("PREDICTED ANSWER:", predicted)
    print("="*100)       


QUESTION: To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
ACTUAL ANSWER: Saint Bernadette Soubirous
PREDICTED ANSWER: It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858

QUESTION: What is in front of the Notre Dame Main Building?
ACTUAL ANSWER: a copper statue of Christ
PREDICTED ANSWER: Next to the Main Building is the Basilica of the Sacred Heart

QUESTION: The Basilica of the Sacred heart at Notre Dame is beside to which structure?
ACTUAL ANSWER: the Main Building
PREDICTED ANSWER: Next to the Main Building is the Basilica of the Sacred Heart


In [10]:
def evaluate(df, n=500):
    correct = 0
    for i in range(n):
        pred = answer_question(df['context'][i], df['question'][i])
        actual = df['answer'][i].lower()
        if actual in pred.lower():
            correct += 1
    return correct / n

accuracy = evaluate(df, 500)
print("Accuracy:", accuracy)

Accuracy: 0.61


In [12]:
context = input("Enter context: ")
question = input("Enter question: ")
answer = answer_question(context, question)
print("\nContext:", context)
print("Question:", question)
print("\nPredicted Answer:", answer)


Context: Super Bowl 50 was won by the Denver Broncos.
Question: Who won Super Bowl 50?

Predicted Answer: Super Bowl 50 was won by the Denver Broncos
